In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Math
import pickle
import os

In [ ]:
# -----------------------------
# Input Directory and file name
# -----------------------------

input_path = "../../Results/Data/"
input_file = os.path.join(input_path, f"results_intermediate.pkl") 

# ------------
# Load Results
# ------------
with open(input_file, 'rb') as f:
    results = pickle.load(f)

# ------------------
# Select theta value
# ------------------
theta_value = list(results.keys())[0]  # First theta, or specify: theta_value = np.pi/2
print(f"Available theta values: {[f'{t:.4f}' for t in results.keys()]}")
print(f"Selected theta = {theta_value:.4f} rad ({np.degrees(theta_value):.2f}°)")
dt = 0.01
N_traj = 1000
site_idx = 1
sample_idx = 1

# -----------------------------
# Output Directory and file name
# -----------------------------

output_path = "../../Results/Plot/Intermediate"  
os.makedirs(output_path, exist_ok=True)


In [ ]:
plt.close('all')

fig01, ax = plt.subplots(figsize=(10,5))

ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], np.real(results[theta_value][dt][N_traj]['lindblad']['rho_list'][:,site_idx,site_idx]), label=r'Lindblad', linewidth=2, linestyle='--')
ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], results[theta_value][dt][N_traj]['anc_trace'][site_idx, :], label=r'Anc_trace', linewidth=2, linestyle=':')
ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], results[theta_value][dt][N_traj]['trajectory_wf']['average_pop'][site_idx, :], label=r'Avg_traj', linewidth=2, alpha=0.5)


ax.set_title(f'Comparison Lindblad, Trace, Avg Trajectories | dt={dt} & N_traj={N_traj} & θ={np.degrees(theta_value):.1f}°')
ax.set_xlabel('Time')
ax.set_ylabel('Population site 1')
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()

save_file = os.path.join(output_path, f"Comparison Lindblad, Trace, Avg Trajectories | dt={dt} & N_traj={N_traj} θ={np.degrees(theta_value):.1f}°.png")
fig01.savefig(save_file, dpi=300, bbox_inches='tight')


plt.show()

In [ ]:
fig02, ax = plt.subplots(figsize=(10,5))

ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], results[theta_value][dt][N_traj]['trajectory_wf']['pop_traj_samples'][site_idx,:,sample_idx], label='Single Traj', linewidth=2)
ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], results[theta_value][dt][N_traj]['trajectory_isolated'][site_idx,:], label='Traj Isolated', linewidth=2, linestyle=':')

ax.set_title(r'Comparison trajectories Collisional vs Isolated')
ax.set_xlabel('Time')
ax.set_ylabel('Population site 1')
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()

save_file = os.path.join(output_path, "Comparison trajectories Collisional vs Isolated.png")
fig02.savefig(save_file, dpi=300, bbox_inches='tight')


plt.show()


In [ ]:
fig03, ax = plt.subplots(figsize=(10, 5))

ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], np.real(results[theta_value][dt][N_traj]['lindblad']['rho_list'][:, site_idx, site_idx]), label=r'Lindblad Site 1', linewidth=2, linestyle=':')
ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], results[theta_value][dt][N_traj]['anc_trace'][site_idx, :], label=r'Trace Ancilla Site 1', linewidth=2, linestyle='--')
ax.plot(results[theta_value][dt][100]['parameters']['times'], results[theta_value][dt][100]['trajectory_wf']['average_pop'][site_idx, :], label=r'Avg Traj N_traj = 100', linewidth=2)
ax.plot(results[theta_value][dt][N_traj]['parameters']['times'], results[theta_value][dt][N_traj]['trajectory_wf']['average_pop'][site_idx, :], label=r'Avg Traj N_traj = 1000', linewidth=2)
ax.plot(results[theta_value][dt][10000]['parameters']['times'], results[theta_value][dt][10000]['trajectory_wf']['average_pop'][site_idx, :], label=r'Avg Traj N_traj = 10000', linewidth=2)

ax.set_xlabel('Time')
ax.set_ylabel('Population')
ax.set_title(f'Comparison different N_traj (dt= {dt})')
ax.legend(fontsize=10)
ax.grid(True)
#ax.set_xlim(30, 45)
#ax.set_ylim(0.18, 0.32)
plt.tight_layout()

save_file = os.path.join(output_path, f"Comparison different N_traj (dt= {dt}).png")
fig03.savefig(save_file, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# =========================================================================
# Comparison different theta value for Single Trajectory vs Isolated System
# =========================================================================
# ----------
# theta list
# ----------
theta_keys = list(results.keys())
num_thetas = len(theta_keys)

# Create figure with stacked subplots (one for each theta)
fig_multi, axes = plt.subplots(num_thetas, 1, figsize=(10, 4 * num_thetas), sharex=True)

# Ensure axes is iterable even if there is only one plot
if num_thetas == 1:
    axes = [axes]

print(f"Plotting comparison for dt={dt}, N_traj={N_traj}, Site={site_idx}, Sample={sample_idx}")

for i, theta in enumerate(theta_keys):
    ax = axes[i]
    
    try:
        # Retrieve data block using global selection variables
        data_block = results[theta][dt][N_traj]
        
        # Extract time and trajectories
        times = data_block['parameters']['times']
        traj_iso = data_block['trajectory_isolated'][site_idx, :]
        traj_sample = data_block['trajectory_wf']['pop_traj_samples'][site_idx, :, sample_idx]
        
        # Plot Sample Trajectory
        ax.plot(times, traj_sample, label=f'Sample Traj #{sample_idx}', color='tab:blue', linewidth=1.5, alpha=0.9)
        
        # Plot Isolated System
        ax.plot(times, traj_iso, label='Isolated System', color='tab:red', linestyle='--', linewidth=2)
        
        # Labels and Title
        deg_value = np.degrees(theta)
        ax.set_title(f'Theta = {theta:.4f} rad ({deg_value:.1f}°) | dt={dt}, N_traj={N_traj}', fontsize=12)
        ax.set_ylabel(f'Population Site {site_idx}')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right', fontsize=9)
        
    except KeyError as e:
        # Handle missing data
        ax.text(0.5, 0.5, f"Data missing: {e}", 
                ha='center', va='center', transform=ax.transAxes, color='red')

# Set X-label only on the bottom plot
axes[-1].set_xlabel('Time')

plt.tight_layout()

# Save figure
save_file_multi = os.path.join(output_path, f"Multi_Theta_Comparison_Iso_vs_Sample_dt{dt}_Ntraj{N_traj}.png")
fig_multi.savefig(save_file_multi, dpi=300, bbox_inches='tight')

plt.show()